# Preprocessing Algorithm for Generation of the SKM Input File for Integration into mitoMammal.ipynb 

## Hansen et al., *Mitochondrial Phosphoproteomes Are Functionally Specialized Across Tissues*

### Maxquant Output- PRIDE Datset PXD030062 (https://www.ebi.ac.uk/pride/archive/projects/PXD030062). Data (proteinGroups.txt) obtained from `combined.zip`

### Mapping Objective 

- The final dataset to be imported into MitoMammal must contain only two columns with no headers: 

1. Ensembl ID
2. Relative Protein Abundance (cumulative peptide counts across six replciates from Hansen et al., study.)

- The Hansen et al., MaxQuant file contains no Ensembl Gene IDs. It contains Gene names and Protein IDs. Therefore these entries must be mapped to Gene IDs, using an intermediary Ensembl Biomart reference file. 
- Both Protein ID and Gene name identifiers in the MaxQuant file are independently mapped to the Protein IDs, Gene names and Ensembl IDs in the Ensembl Biomart reference file. 
- The result is then mapped onto the Ensembl IDs in the MitoMammal final tsv, obtained from the author repository. 
-The final output file therefore links the Ensembl Gene ID to its corresponding relative protein abuncance value, with all confirmed, intermediary information removed.
_____________________________________________________________

### Required Input Files 

To run this mapping algorithm, the following input files are required: 

1. **Ensembl BioMart Export for the Species of Interest**
- In this context, the species is *Mus musculus* (mouse). The BioMart export file should select for the following: 
  - Gene Stable ID
  - Gene name 
  - UniProtKB/Swiss-Prot ID
  - UniProtKB/TrEMBL ID 
- These identifiers provide the reference mapping required to link Ensembl IDs, Gene names and Protein IDs across the three datasets. 
  
2. **Final Processed MitoMammal TSV Files (`heart.tsv` and `BAT.tsv`)**
 - The final processed TSV files from the MitoMammal repository contain Ensembl IDs and the corresponding relative protein abundance value, specific to heart or BAT tissue. These files are used as reference datasets to confirm & reproduce the author's method of data processing and indentifier mapping, to create the SKM dataset for integration into MitoMammal. From this file the Ensembl ID column will retained, but with corresponding peptide counts specific for SKM, obtained from the Maxquant file. The purpose thereof, is to deviate as little as possible from the author's final file format, that is subsequently called into MitoMammal. 
 - The files are available at: https://gitlab.com/habermann_lab/mitomammal/-/tree/main/data/Proteomic?ref_type=heads
  
3. **Hansen et al., MaxQuant `proteinGroups.txt` file**
- The `proteinGroups.txt` file generated by MaxQuant output of LC-MS/MS from the Hansen et al., study is the primary proteomic dataset being mapped. This file does not contain the Ensembl IDs required for the final dataset that is called into MitoMammal. The file does however contain Gene names and Protein IDs. Both the Gene names and Protein IDs will be mapped, to confirm that the correct peptide count value is assigned to the correct Ensembl Gene ID. Where one identifier cannot be mapped, the other will serve as alternative mapping route.   



In [19]:
# Begin Mapping with the File that Contains the most Identifier entries 

# Gene Name Mapping: Ensembl Biomart and Hansen Protein MaxQuant output


### Files Used/ Created in this Section: 

**Source Tables**
- Hansen MaxQuant: `proteinGroups.xlsx`, only concerning the Gene names, Protein IDs and the three Sum_Peptides columns (the sum columns were made manually in the excel file).
  -  **7632** rows (protein groups) each with a Protein ID, of which 7165 carry a Gene name and   **467** rows have no Gene name entry. 

- `BioMart`: The mouse mapping TSV file containing the whole mouse genome including Gene stable ID, Gene name, Swiss-Prot and TrEMBL Protein IDs. 


  
**Preliminary Count Checks**
- ensembl_ids_bat and ensembl_ids_heart: set of Ensembl IDs from the MitoMammal reference TSVs, used to establish the size of the dataset (number of Ensembl IDs). 
  - Both the Heart and BAT files contain **6652** Gene IDs, which sets the anticipated file size for the final SKM dataset. 



**Exploratory audit files** 
1. `gene_protein_id_intersection.xlsx`: Within the Hansen MaxQuant file, checking which rows contain both a Gene name and a Protein ID or only one of either. All rows in the file contain a Protein ID. 
- Sheets in the xlsx:
   - `Both_Gene_Prot`: gene name and protein ID both present
   - `geneName_only`:  gene name present, protein ID blank (**No hits in this file**)
   - `protID_only`:    protein ID present, gene name blank (**467 rows**)
  
2. `ensembl_biomart_identifier_combinations.xlsx`: Within the Ensembl BioMart file, checking which identifiers BioMart holds per Gene ID.
- Sheets in the xlsx:
   - `geneName_SwissProt_TrEMBL`: all three identifiers present (**3354 rows**)
   - `geneName_SwissProt`:        gene name and Swiss-Prot, TrEMBL blank (**13 063 rows**)
   - `geneName_TrEMBL`:           gene name and TrEMBL, Swiss-Prot blank (**39 141 rows**)
   - `SwissProt_TrEMBL`:          both accessions, gene name blank (**No hits in this file**)
   - `geneName_only`:             gene name only, neither accession (**14 857 rows**)
   - `SwissProt_only`:            Swiss-Prot only (**No hits in this file**)
   - `TrEMBL_only`:               TrEMBL only (**51 rows**)
   - `GeneID_only`:               Ensembl gene ID with no mapping identifier at all (**43 rows**)
 


**Working Files for Mapping**
1. `Hansen_gene_mapping`: Copy of Hansen with the Gene names column split on semicolons into Python lists
  
2. `gene_lookup`: dictionary, {cleaned gene name: [Hansen row numbers]}. 
   - Keys are stripped, lowercased and de-hyphenated so Hansen's older style (Mtnd1) matches BioMart's current style (mt-Nd1)
    -  **7094** unique gene names, of which **293** appear in more than one Hansen row.
    -   That repetition is the one-to-many effect of isoforms and paralogs sharing a parent gene, not a duplication error.

3. `BioMart_gene_mapping`: Copy of BioMart with an added `gene_name` column, also stripped, lowercased and de-hyphenated.

4. `matched_gene_names` / `matched_gene_names_df`:
   
 - The pairs table built as a skeleton and then widened:
   - Hansen's `Gene names` column holds lists, and pandas cannot merge a list against BioMart's single string. 
   - `gene_lookup` unpacks the lists and returns row numbers (indeces), so a paired table of row numbers can be built: `BioMart_idx` and `Hansen_idx`. 
   - **One row per match event**: A Hansen row generates one entry per BioMart partner. 

  
  - Widened by two merges:
    - Using `left_on=<idx column>, right_index=True` to treat the stored number as a row position and pull that row's columns across BioMart's identifiers first, then Hansen's gene names, protein IDs and peptide counts. 
  


**Output Table**
- `Hansen_BioMart_GeneNames`: Every original Hansen row, left-merged against `matched_gene_names_df` on the `Hansen_idx` pointers that `gene_lookup` recorded, in addition to the `Gene name source` column.
  -The lookup stored, for each cleaned gene name, the row numbers it came from and those pointers became the pairs table. 
  - The merge traces back the pointers to reattach the matches to the original file and append the row's information. 
  - The merge is left on the Hansen file, so the 391 rows with no Gene name matches to the BioMart file are retained. 



**Final Excel File**
1. `gene_name_mapping_Hansen_BioMart.xlsx`:

- Sheets in the xlsx:
   - `All Hansen rows`:              every row of the merged table and a summary of the three sheets below (**30 219 rows**)
   - `Matched Hansen and BioMart`:   mapped successfully on Gene name (**29 361 rows/ match events, 6 774 distinct Hansen rows**)
   - `Hansen MaxQuant only`:         Gene name present in Hansen, no BioMart match (**391 rows**)
   - `No gene name in Hansen`:       blank Hansen Gene name cells, Protein IDs are displayed instead (**467 rows**)





  






In [20]:
import csv

In [21]:
# Check how many Ensembl IDs are in the BAT tsv (MitoMammal Reposiroty) contains

with open(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\Files used to verify method of integration\BAT_data.tsv") as f:
    reader = csv.reader(f, delimiter="\t")
    ensembl_ids_bat = {row[0] for row in reader}
    print("No. Ensembl Gene IDs in MitoMammal BAT file:", len(ensembl_ids_bat))

No. Ensembl Gene IDs in MitoMammal BAT file: 6652


In [22]:
#  Check how many Ensembl IDs are in the heart tsv (MitoMammal Repository)

with open(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\Files used to verify method of integration\Heart_data.tsv") as f:
    reader = csv.reader(f, delimiter="\t")
    ensembl_ids_heart = {row[0] for row in reader}
    print("No. Ensembl Gene IDs in MitoMammal heart file:", len(ensembl_ids_heart))

    # Use either 'with open' or pandas


No. Ensembl Gene IDs in MitoMammal heart file: 6652


In [23]:
import pandas as pd

In [24]:
# Check how many Ensembl IDs are in the Ensembl Mouse Mapping TSV

df= pd.read_csv(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\Files used to verify method of integration\mouse ID mapping.tsv", sep="\t")
print("No. Ensembl Gene IDs in Ensembl BioMart Mapping file:", df['Gene stable ID'].count())

# This is the whole mouse genome pulled from Biomart. 
# I cannot select only for MT on the chromosome region, because many mitochondrial
# proteins are encoded by nuclear DNA. 

No. Ensembl Gene IDs in Ensembl BioMart Mapping file: 70509


In [25]:
# Check how many Gene names and Protein IDs are in Maxquant file

df=pd.read_csv(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\Files used to verify method of integration\proteinGroups.txt",sep="\t")
print(f"No. of Protein IDs: {df['Protein IDs'].count()}")
print(f"No. of Gene names: {df['Gene names'].count()}")

# .count because I do not want the code to consider blank cells


No. of Protein IDs: 7632
No. of Gene names: 7165


C:\Users\cayle\AppData\Local\Temp\ipykernel_23224\953827961.py:3: DtypeWarning: Columns (626,627) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\Files used to verify method of integration\proteinGroups.txt",sep="\t")


#### Determine which protein IDs have no corresponding Gene name in the Hansen Maxquant file


In [26]:
# This means that for 467 Protein IDs have no corresponding Gene name. 
# Output this list. 
df = pd.read_excel(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\Files used to verify method of integration\proteinGroups.xlsx")

df = df[['Gene names','Protein IDs', 'Sum_Peptides BAT', 'Sum_Peptides H', 'Sum_Peptides SKM' ]]


both_gene_prot = []
geneName_only = []
protID_only = []

# .iterrows is pandas method that loops through dataframe
# row by row and outputs the row number (index) and the row data

# Go through spreadsheet one row at a time. top to bottom. 
# For each rowm check whether has both gene name and protein ID. 
# If it does, save the whole row into the both_gene_prot list.
# But if that was not true, if the row has only a gene name, but the protein ID cell is blank, check for that instead. 
# Save that whole row in geneName_only. 
# Otherwise, check if the opposite is true - Protein ID only but no Gene name.
# If found, save that whole row in protID_only list. 

for index, row in df.iterrows(): 
    if pd.notna(row['Gene names']) and pd.notna(row['Protein IDs']): # .notna() alone will work through a whole column at once, but .notna('XYZ') checks each cell. '&' is for whoel column, 'and' is for row-by-row iteration. 
        both_gene_prot.append(row) #indented twice, because it is inside the for loop and inside the if. This line only runs if the above if statement is true.
    elif pd.notna(row['Gene names']) and pd.isna(row['Protein IDs']):
        geneName_only.append(row)
    elif pd.notna(row['Protein IDs']) and pd.isna(row['Gene names']):
        protID_only.append(row)

both_df = pd.DataFrame(both_gene_prot)
geneName_only_df = pd.DataFrame(geneName_only)
protID_only_df = pd.DataFrame(protID_only)
# The code above converts the outputs to a dataframe table with headers, 
# to be exported as an excel file 

# Make into one Excel file with three separate sheets
with pd.ExcelWriter(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\gene_protein_id_intersection.xlsx") as writer:
    both_df.to_excel(writer, sheet_name="Both_Gene_Prot", index=False)
    geneName_only_df.to_excel(writer,sheet_name="geneName_only", index=False)
    protID_only_df.to_excel(writer,sheet_name="protID_only", index=False)




#### Determine which mapping identifiers (Gene name, UniProtKB/Swiss-Prot ID, UniProtKB/TrEMBL ID) can be confirmed for each (Ensembl) Gene ID 

In [27]:
df = pd.read_csv(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\Files used to verify method of integration\mouse ID mapping.tsv", sep="\t")

geneName_SwissProt_TrEMBL = [] # all three identifiers present for one Ensembl ID

geneName_SwissProt = []
geneName_TrEMBL = []
SwissProt_TrEMBL = []

geneName_only = []
SwissProt_only = []
TrEMBL_only = []

GeneID_only = [] # This means there is are no other mapping identifiers, and the Ensembl ID is present only 


for index, row in df.iterrows():
    if pd.notna(row['Gene name']) and pd.notna(row['UniProtKB/Swiss-Prot ID']) and pd.notna(row['UniProtKB/TrEMBL ID']):
        geneName_SwissProt_TrEMBL.append(row)

    elif pd.notna(row['Gene name']) and pd.notna(row['UniProtKB/Swiss-Prot ID']):
        geneName_SwissProt.append(row)

    elif pd.notna(row['Gene name']) and pd.notna(row['UniProtKB/TrEMBL ID']):
        geneName_TrEMBL.append(row)

    elif pd.notna(row['UniProtKB/Swiss-Prot ID']) and pd.notna(row['UniProtKB/TrEMBL ID']):
        SwissProt_TrEMBL.append(row)

    elif pd.notna(row['Gene name']) and pd.isna(row['UniProtKB/Swiss-Prot ID']) and pd.isna(row['UniProtKB/TrEMBL ID']):
        geneName_only.append(row)

    elif pd.notna(row['UniProtKB/Swiss-Prot ID']) and pd.isna(row['Gene name']) and pd.isna(row['UniProtKB/TrEMBL ID']):
        SwissProt_only.append(row)

    elif pd.notna(row['UniProtKB/TrEMBL ID']) and pd.isna(row['Gene name']) and pd.isna(row['UniProtKB/Swiss-Prot ID']):
        TrEMBL_only.append(row)
        
    elif pd.isna(row['Gene name']) and pd.isna(row['UniProtKB/Swiss-Prot ID']) and pd.isna(row['UniProtKB/TrEMBL ID']):
        GeneID_only.append(row)

    

geneName_SwissProt_TrEMBL_df = pd.DataFrame(geneName_SwissProt_TrEMBL)

geneName_SwissProt_df = pd.DataFrame(geneName_SwissProt)
geneName_TrEMBL_df = pd.DataFrame(geneName_TrEMBL)
SwissProt_TrEMBL_df = pd.DataFrame(SwissProt_TrEMBL)

geneName_only_df = pd.DataFrame(geneName_only)
SwissProt_only_df = pd.DataFrame(SwissProt_only)
TrEMBL_only_df = pd.DataFrame(TrEMBL_only)

GeneID_only_df = pd.DataFrame(GeneID_only)


with pd.ExcelWriter(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\ensembl_biomart_identifier_combinations.xlsx") as writer:
    geneName_SwissProt_TrEMBL_df.to_excel(writer, sheet_name="geneName_SwissProt_TrEMBL", index=False)
    geneName_SwissProt_df.to_excel(writer, sheet_name="geneName_SwissProt", index=False)
    geneName_TrEMBL_df.to_excel(writer, sheet_name="geneName_TrEMBL", index=False)
    SwissProt_TrEMBL_df.to_excel(writer, sheet_name="SwissProt_TrEMBL", index=False)
    geneName_only_df.to_excel(writer, sheet_name="geneName_only", index=False)
    SwissProt_only_df.to_excel(writer, sheet_name="SwissProt_only", index=False)
    TrEMBL_only_df.to_excel(writer, sheet_name="TrEMBL_only", index=False)
    GeneID_only_df.to_excel(writer, sheet_name="GeneID_only", index=False)               



In [28]:
import pandas as pd

In [29]:

# Hansen Maxquant file: Gene names','Protein IDs', 'Sum_Peptides BAT', 'Sum_Peptides H', 'Sum_Peptides SKM'
# Ensembl BioMart file: Gene stable ID	Gene name	UniProtKB/Swiss-Prot ID	UniProtKB/TrEMBL ID

# Map Gene Names First. 
# A left merge on the Maxquant file, because the Ensembl Biomart file is the whole mouse genome, not specific to mitochondria. 
Hansen = pd.read_excel(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\Files used to verify method of integration\proteinGroups.xlsx")
BioMart = pd.read_csv(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\Files used to verify method of integration\mouse ID mapping.tsv", sep="\t")

Hansen = Hansen[['Gene names','Protein IDs', 'Sum_Peptides BAT', 'Sum_Peptides H', 'Sum_Peptides SKM']]

# Every cell that contains multiple gene names (isoforms,paralogs) separated by a semicolon, must be converted into a Python list. 
# This is so that the file retains the original number of rows; nothing is exploded. 

Hansen_gene_mapping = Hansen.copy()

# Make a copy of the Hansen table, no alterations to the original 
# Take the "Gene names" column and ascertain that each cell is treated as text, it was likely object dtype before, meaning treated as mixed or non-numeric.
# Split each cell where a semicolon appears, turning it into a Python list
# Place the lists back into the "Gene names" column, replacing the object dtype entries that were there previously. 

Hansen_gene_mapping["Gene names"]= (
    Hansen_gene_mapping["Gene names"].astype("string")
.str.split(";")
)

### THE DICTIONARY: 

# The dictionary is being made from the Hasnsen file, because of the multiple gene entries per cell.
# Therefore, each cell cannot be matched one-to-one as is.2
# Entries need to be 'looked up' and the index of where that entry originated from is recorded

gene_lookup = {} #curly brackets creates an empty dictionary 

# Go through the "Gene names" column one cell at a time, keeping track of both the row number and cell contents 
# Check whether the cell contains a list
# If not (blank cell, & cells with single entries are treated as single item lists), skip this row and move to the next

for idx, gene_names in Hansen_gene_mapping["Gene names"].items():
    if not isinstance(gene_names, list):
        continue
    for gene in gene_names:
        # This is a nested for loop. The outer for loop iterates through the rows, 
        # whilst this inner loop moves through each entry within a cell's single/multiple item list.
        # The index therefore stays at that specific row number. 
        gene = gene.strip().lower().replace("-", "")
        # This is so that leading and trailing blank spaces are no concern in python. 
        # The above code is necessary, because Python compares character by character, so COA4 and Coa4 will be treated as different strings. 
        # This means the same gene, formatted differently will not map. 
        # To fix this, the gene names in the lookup dictionary are changed to lowercase. 
        # When the Biomart file is mapped on, the gene names will also all be converted to lowercase.
        # .replace("-", ""): removes hyphens so Hansen's older style (Mtnd1) matches BioMart's current style (mt-Nd1), the same way .lower() removes case differences.
        if gene not in gene_lookup:
            gene_lookup[gene] = [idx]
        else:
            gene_lookup[gene].append(idx)

            # The above code says: Check if the current gene is in the dictionary,
            # if not, add as new entry. 
            # If it is, add the row number/index to the list already stored in that gene name. 
            # So if a gene name appears in several rows the index will look like [340, 453,764]


print("Unique gene names in lookup:", len(gene_lookup))

print("Gene names that appear in more than one row:", sum(1 for v in gene_lookup.values() if len(v) >1))

# gene_lookup.values(): all the values in the dictionary (the lists of row numbers), ignoring the keys
# for v in: go through them one at a time, calling each one 'v'
# if len(v) > 1: only keep the ones whose list holds more than one row number
# 1: produce the number 1 for every item that passes the test
# sum(...): add up all those 1s, which gives the count



Unique gene names in lookup: 7094
Gene names that appear in more than one row: 293


In [30]:
# 292 is the number of Gene names that appear more than once, not the number of appearances.
# More than one protein ID carries the same gene name in the Hansen Maxquant file. Isoforms and paralogs.

In [31]:
BioMart_gene_mapping = BioMart.copy()
BioMart_gene_mapping["gene_name"] = BioMart_gene_mapping["Gene name"].str.strip().str.lower().str.replace("-", "", regex=False)

# Created a new column "gene_name" that holds the lowercase, unhyphenated, stripped BioMart gene name. 
# This is so that the Original Gene name column can be retained. 
# The gene_name is what will be used for matching against the lookup dictionary.  
# Create a list that contains the gene name matches between the Hansen and BioMart file. 
# This will serve as a recclrd of which BioMart row is matched to which Hansen row. 
matched_gene_names = []

# Go through BioMart gene_name column one cell at a time, recording both row number and gene name. 
# If the cell is blank, skip and continue
# Check if the BioMart gene key is in the Hansen lookup dictionary
# If it is, go through each Hansen row number stored under that same key, one at a time (recording instances of multiple appearances of same gene name)
# For each instance, record the index of the BioMart row and the Hansen row where each gene name in the match pair came from. 

for BioMart_idx, gene_name in BioMart_gene_mapping["gene_name"].items():
    if pd.isna(gene_name):
        continue
    if gene_name in gene_lookup:
        for Hansen_idx in gene_lookup[gene_name]:
            matched_gene_names.append({"BioMart_idx": BioMart_idx, "Hansen_idx": Hansen_idx})


# Take matched_gene_names_df (the pairs table holding BioMart_idx and Hansen_idx) and merge new columns onto it
# Fetch those columns from BioMart_gene_mapping, taking only Gene stable ID, Gene name, UniProtKB/Swiss-Prot ID and UniProtKB/TrEMBL ID
# Use the number in matched_gene_names_df's BioMart_idx column as a row position in BioMart_gene_mapping
# Keep every pair already in matched_gene_names_df, even if no BioMart row is found
matched_gene_names_df = pd.DataFrame(matched_gene_names)
matched_gene_names_df = matched_gene_names_df.merge(
    BioMart_gene_mapping[["Gene stable ID", "Gene name"]],
    left_on = "BioMart_idx",
    right_index=True,
    how="left"
)

print(matched_gene_names_df.shape)
print(matched_gene_names_df.head(20))

(29361, 4)
    BioMart_idx  Hansen_idx      Gene stable ID Gene name
0             0        2685  ENSMUSG00000064341    mt-Nd1
1             1        2686  ENSMUSG00000064345    mt-Nd2
2             2        2656  ENSMUSG00000064351    mt-Co1
3             3        2657  ENSMUSG00000064354    mt-Co2
4             4        2690  ENSMUSG00000064356   mt-Atp8
5             5        2665  ENSMUSG00000064357   mt-Atp6
6             6        2658  ENSMUSG00000064358    mt-Co3
7             7        2687  ENSMUSG00000064360    mt-Nd3
8             9        2688  ENSMUSG00000064363    mt-Nd4
9            10        2689  ENSMUSG00000064367    mt-Nd5
10           93        3801  ENSMUSG00000069036       Sry
11          185        4435  ENSMUSG00000069045     Ddx3y
12          186        4435  ENSMUSG00000069045     Ddx3y
13          187        4435  ENSMUSG00000069045     Ddx3y
14          228        7272  ENSMUSG00000069049   Eif2s3y
15          229        7272  ENSMUSG00000069049   Eif2s3y
16 

In [32]:
# Take the matched_gene_names_df just created (Now retains all the BioMart columns) and merge Hansen columns onto it
# Fetch the columns from the Hansen File, retaining only Gene names, Protein IDs, Sum_Peptides BAT, Sum_Peptides H and Sum_Peptides SKM
# Use the index matched in the matched_gene_names_df's Hansen idx columnn as pointer to row position in the Hansen file

matched_gene_names_df = matched_gene_names_df.merge(
    Hansen[[ "Gene names", "Protein IDs", "Sum_Peptides BAT", "Sum_Peptides H", "Sum_Peptides SKM"]],
   left_on="Hansen_idx",
   right_index=True,
   how="left" 
)

# Now I am going to merge the full, original Hansen table and merge the BioMart columns onto it. Every original Hansen row will be kept.
# This will allow me to see which Hansen Gene names do not match to Gene names in Biomart. 
# The columns merged onto the OG Hansen file are "Hansen_idx", "Gene stable ID", "Gene name", "UniProtKB/Swiss-Prot ID", "UniProtKB/TrEMBL ID"
# The row position in the Hansen_idx will be traced and matched to that in the OG file
# Retain every Hansen row (OG) even if no BioMart match was recorded. Unmapped rows will return blank BioMart columns.

Hansen_BioMart_GeneNames = Hansen.merge(
    matched_gene_names_df[["BioMart_idx", "Hansen_idx",  "Gene stable ID", "Gene name"]],
    left_index=True,
    right_on="Hansen_idx",
    how="left"
)

#Unmatched rows have no BioMart row, so BioMart_idx is blank there, which forces decimals.
# "Int64" (capital I) keeps whole numbers while allowing blanks.
Hansen_BioMart_GeneNames["BioMart_idx"] = Hansen_BioMart_GeneNames["BioMart_idx"].astype("Int64")

# Now I need to create the Gene Name source column
# I will use if and elif statements for this 

gene_name_source = []
# Gene names (s) is Hansen Maxquant
# Gene name is BioMart file 

for index, row in Hansen_BioMart_GeneNames.iterrows():
    if pd.isna(row["Gene names"]): # Where in Hansen File are there no Gene names?
        gene_name_source.append("No gene name in Hansen MaxQuant")
    elif pd.notna(row['Gene names']) and pd.notna(row['Gene name']):
        gene_name_source.append("Hansen MaxQuant, BioMart")
    elif pd.notna(row["Gene names"]) and pd.isna(row['Gene name']):
        gene_name_source.append("Hansen MaxQuant Only")

Hansen_BioMart_GeneNames["Gene name source"] = gene_name_source

# There are no BioMart-only rows in this table, because the Hansen MaxQuant file is the reference:
# BioMart covers the whole mouse genome and is not mitochondria-specific, so genes never detected
# in the experiment are excluded here and kept on a separate sheet instead.
#
# Each identifier is mapped independently, one at a time, because a given Hansen row may match
# BioMart on one identifier but not another, so a gene name may fail where a protein ID successfully maps, or the vice versa.
#  Mapping them separately keeps these outcomes visible, so a failed mapping can be
# traced to the specific identifier that failed rather than being hidden inside a single combined result.

print(len(gene_name_source), len(Hansen_BioMart_GeneNames))
print(Hansen_BioMart_GeneNames["Gene name source"].value_counts())
print("Distinct Hansen rows matched:", matched_gene_names_df["Hansen_idx"].nunique())

matched_bm_rows = set(matched_gene_names_df["BioMart_idx"])
BioMart_only = BioMart_gene_mapping[BioMart_gene_mapping.index.isin(matched_bm_rows)]


30219 30219
Gene name source
Hansen MaxQuant, BioMart           29361
No gene name in Hansen MaxQuant      467
Hansen MaxQuant Only                 391
Name: count, dtype: int64
Distinct Hansen rows matched: 6774


In [33]:
Hansen_BioMart_GeneNames[[
    "BioMart_idx",
    "Hansen_idx",
    "Gene stable ID",
    "Gene names",
    "Gene name",
    "Gene name source"
]].head(10)

,BioMart_idx,Hansen_idx,Gene stable ID,Gene names,Gene name,Gene name source
7645.0,23969,0,ENSMUSG00000031174,Rpgr,Rpgr,"Hansen MaxQuant, BioMart"
7647.0,23972,0,ENSMUSG00000031174,Rpgr,Rpgr,"Hansen MaxQuant, BioMart"
7655.0,23989,0,ENSMUSG00000031174,Rpgr,Rpgr,"Hansen MaxQuant, BioMart"
7659.0,23993,0,ENSMUSG00000031174,Rpgr,Rpgr,"Hansen MaxQuant, BioMart"
7660.0,23994,0,ENSMUSG00000031174,Rpgr,Rpgr,"Hansen MaxQuant, BioMart"
7663.0,23996,0,ENSMUSG00000031174,Rpgr,Rpgr,"Hansen MaxQuant, BioMart"
7664.0,23999,0,ENSMUSG00000031174,Rpgr,Rpgr,"Hansen MaxQuant, BioMart"
7672.0,24008,0,ENSMUSG00000031174,Rpgr,Rpgr,"Hansen MaxQuant, BioMart"
9209.0,27198,1,ENSMUSG00000031137,Fgf13,Fgf13,"Hansen MaxQuant, BioMart"
9213.0,27203,1,ENSMUSG00000031137,Fgf13,Fgf13,"Hansen MaxQuant, BioMart"


In [34]:
# Define the four columns every sheet will show, so the format is identical throughout
gene_source_cols = ["BioMart_idx", "Hansen_idx", "Gene stable ID", "Gene names", "Gene name", "Gene name source"]

# Sheet 4 uses a different column set, since its gene name columns are empty by definition
no_gene_name_cols = ["Hansen_idx", "Protein IDs", "Gene name source"]

# Sheet 1: every Hansen row, trimmed to the audit columns. This is a summary file of the other 3 sheets. 
Hansen_BioMart_audit = Hansen_BioMart_GeneNames[gene_source_cols]

# Sheet 2: only the rows that matched in both files
matched_audit = Hansen_BioMart_GeneNames[
    Hansen_BioMart_GeneNames["Gene name source"] == "Hansen MaxQuant, BioMart"][gene_source_cols]

# Sheet 3: gene name present in Hansen but not found in BioMart
maxquant_only_audit = Hansen_BioMart_GeneNames[
    Hansen_BioMart_GeneNames["Gene name source"] == "Hansen MaxQuant Only"][gene_source_cols]

# Sheet 4 needs different columns: these rows have an empty gene name cell in the Hansen MaxQuant file, so the Hansen Protein IDs are the only identifier available for them.
# No BioMart match was attempted for these rows, because there was no gene name to look up.
# These will be matched using the protein IDs from both files. 
no_gene_name_audit = Hansen_BioMart_GeneNames[
    Hansen_BioMart_GeneNames["Gene name source"] == "No gene name in Hansen MaxQuant"][no_gene_name_cols]

with pd.ExcelWriter(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\gene_name_mapping_Hansen_BioMart.xlsx") as writer:
    Hansen_BioMart_audit.to_excel(writer, sheet_name="All Hansen rows", index=False)
    matched_audit.to_excel(writer, sheet_name="Matched Hansen and BioMart", index=False)
    maxquant_only_audit.to_excel(writer, sheet_name="Hansen MaxQuant only", index=False)
    no_gene_name_audit.to_excel(writer, sheet_name="No gene name in Hansen", index=False)

In [35]:
# In the Hansen file is : 
# No. of Protein IDs: 7632
# No. of Gene names: 7165

# In MitoMammal Dataset: 
# No. Ensembl Gene IDs in MitoMammal heart file: 6652

# BioMart and Hansen File:
# 6774 Unique Gene names macthes accounting for formatting differences between the file gene names

# "No gene name in Hansen Maxquant":
# This means for that row, the Gene name column is returned as empty. 
# This number is 467

# "Hansen MaxQuant Only":
# This mean that row contains a gene name, but it did not match to any gene names present in the BioMart file. 
# This numner is 391


_______________________________________________________________________________

___________________________________________________________________________________________________

# Protein ID Mapping: Ensembl Biomart and Hansen Protein MaxQuant output

### Files Used/ Created in this Section: 

This section retains most of the Gene Mapping structure. The Hansen MaxQuant's 'Majority protein IDs' column is macthed agaisnt both BioMart accession columns, Swiss-Prot and TrEMBL, because MaxQuant does not specify which database the accession originates from. A single cell in the Hansen file can hold both Swiss-Prot and TrEMBL accessions.


**Source Tables Reloaded**
- `Hansen`: reloaded to include `Majority protein IDs`. 
  - 7 632 rows with **zero blank cells**, so every protein group carries at least one accession.
  - The `Majority protein IDs` column holds **17 810 individual accessions** across those 7 632 rows, with the largest single group containing **34**.
  - Each row is one protein group, and a group holds every accession whose peptides could not be distinguished from the leading (first accession/Protein ID in cell) protein's.
  - A trailing (in the second position or any position thereafter) accession is retained in `Majority protein IDs` only if it shares at least half the peptides identified for the leading protein. If it is below this threshold it is dropped.
  
    - The `Protein IDs` column holds **23 338 accessions** across the same 7 632 rows, largest group **36**.
    - This is the unfiltered version of the same groupings.
    - The **5 528 difference** between the two columns is what MaxQuant's 50% peptide-evidence filter
    - The weakly supported accessions can be excluded from mapping, as an accession can belong to a
    different gene and would carry the group's peptide count to an Ensembl ID whose protein was never confidently present in the sample.
    
- Mapping on `Majority protein IDs` rather than `Protein IDs` is therefore a deliberate choice, opting for less coverage but at a lower false-assignment rate.



**Working Files for Mapping**
1. `Hansen_protein_mapping`: Copy of Hansen with 'Majority protein IDs' column split on semicolons into Python lists
  
2. `protein_lookup`: dictionary, {Protein ID/accession: [Hansen row numbers]}. 
  - Protein IDs are stripped, but not lowercased or de-hyphenated, since Protein IDs are always uppercase and I have chosen not to strip Protein ID suffixes at this point. 
  - **17 810 unique accessions**, none appearing in more than one **Hansen row**. 
  - This is in obvious contrast to the gene name lookup's **293 repeats**, confirming accessions are a cleaner one-to-one identifier on the Hansen side.  

3.  `protein_lookup_all`: the same lookup built on the full `Protein IDs` column: **23 338 unique accessions** , again zero repeats. This was for curiosity and comparison only. This file was not used for mapping.

4. `BioMart_protein_mapping`: Copy of BioMart with an added `SwissProt_key` and `TrEMBL_key`, stripped. 

5. `matched_protein_ids` / `matched_protein_ids_df`:

- The pairs table, built and widened in the same way as the Gene Mapping section, with an addition.
   - `Matched on`: A third column naming which BioMart column produced a match with a Hansen Protein ID, on "Swiss-Prot ID" or "TrEMBL ID". 
   - The loop tests the Swiss-Prot ID" and "TrEMBL ID" columns separately, and appends independently in two separate rows.
   - A Protein ID match on both accession types will therefore be recorded in two rows, labelled each way, and not yet as a combined entry (baby steps).
   - "Both" is a property of a Hansen row, not of the match event and is identified by `Protein_Identifier`, which grous the accession labels (Swiss/TrEMBL) for each  `Hansen_idx` into a single set. 

  - One new row is generated per match event and a Hansen row generates one entry per BioMart match (recalling that a single Hansen cell can contain up 34 accessions in the 'Majortiy protein IDs column).
    - The results:
      - **13 932** match events
      - **7 307 TrEMBL**
      - **6 625 Swiss-Prot**
      - Only **7 185 unique Hansen rows**
      - 13 873 unique BioMart rows.

  - Widened by two merges:
    - Using `left_on= idx column, right_index=True` to treat the stored number as a row position and pull that row's columns across.
  
6. `Prot_Identifier`:
  - A Series indexed by `Hansen_idx`, holding the set of `Matched on` labels collected across every match event belonging to that Hansen row.
  - This step converts the individual match events into a match classification for each Hansen MaxQuant row. 
  - Three match classifications are possible:
     - "Swiss-Prot ID" (only)
     - "TrEMBL ID" (only)
     - "Swiss-Prot ID", "TrEMBL ID" (both)
  
  


**Output Table**:
*Master output table**
- `Hansen_BioMart_ProteinIDs`: every original Hansen row, left-merged against `matched_protein_ids_df` on the `Hansen_idx` pointers that `protein_lookup` recorded, plus the additonal `Protein ID source` column.
- The merge is left, so the 447 rows whose accessions found nothing in BioMart survive with blank BioMart columns.
- `Hansen_BioMart_ProteinIDs` holds **14 379 rows**, made up of the 13 932 match events plus the 447 Hansen rows that matched nothing and were kept by the left merge. 
- This is not a count of proteins: the 7 632 Hansen protein groups inflate to 14 379 rows because one accession can sit at several Ensembl genes.
- This duplication is biologically real and not a mapping defect. 

Outcome per Hansen row, total 7632:
  - Matched on both: **2801**
  - Swiss-Prot only: **3272**
  - TrEMBL only: **1112**
  - No match in BioMart **447**


**Breakdown of the 447 unmatched rows:**

All 447 rows were labelled Hansen MaxQuant Only, meaning no exact BioMart protein-ID match was found.


- Contaminants: 35 rows
  - Identified by the CON__ prefix.
  - These are proteins from non-mouse sources, such as keratin, trypsin, and bovine albumin.
  - They are not expected to have a mouse Ensembl gene ID, so their failure to match is expected.
- Decoys: 266 rows
  - Identified by the REV__ prefix.
  - These are reversed protein sequences generated by MaxQuant to estimate the false discovery rate (FDR).
  - They do not represent real proteins, so their failure to match is expected.
- Real unmatched proteins: 146 rows
  - These are the remaining rows that are neither contaminants nor decoys.
  - These were investigated further to determine why they did not match BioMart.
    - Isoform suffix failures: 106 rows
- Absent from BioMart: 40 rows
  - The protein accession, including its base form, could not be found in the BioMart export.
  - These represent the genuine protein-ID mapping failures.

**Final Excel file**

1. `protein_id_mapping_Hansen_BioMart.xlsx`

Columns on every sheet: Hansen_idx, Gene stable ID, Majority protein IDs, UniProtKB/Swiss-Prot ID,
UniProtKB/TrEMBL ID, Protein ID source.

- Sheets in the xlsx:
   - `All Hansen rows`:            Every row of the merged table (**14 379 rows**)
   - `Matched Hansen and BioMart`: The three successful values of `Protein ID source`, selected with `.isin()` rather than `==` since more than one label qualifies (**13 932 rows, 7 185 distinct Hansen rows**)
      - `Hansen MaxQuant, BioMart (Swiss-Prot and TrEMBL)`: The accession was found in both BioMart columns (**8 211 rows, 2 801 distinct Hansen rows**)
      - `Hansen MaxQuant, BioMart (Swiss-Prot)`: Swiss-Prot only (**3 458 rows, 3 272 distinct Hansen rows**)
      - `Hansen MaxQuant, BioMart (TrEMBL)`: TrEMBL only (**2 263 rows, 1 112 distinct Hansen rows**)
   - `Hansen MaxQuant only`:       All unmatched rows and the four sheets below split these by cause (**447 rows**)
   - `Contaminants`:               `CON__` prefixed, correctly unmatched (**35 rows**)
   - `Reverse decoys`:             `REV__` prefixed, correctly unmatched (**266 rows**)
   - `Isoform suffix failures`:    Base accession present in BioMart, suffix blocked the match (**106 rows**)
   - `Absent from BioMart`:        Genuine mapping failures (**40 rows**)




  






In [36]:
# Check how many Protein IDs there are in both Swiss and TrEMBL columns in the Ensembl BioMart File

df= pd.read_csv(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\Files used to verify method of integration\mouse ID mapping.tsv", sep="\t")
print("No. Ensembl Swiss Prot IDs in Ensembl BioMart Mapping file:", df['UniProtKB/Swiss-Prot ID'].count())
print("No. Ensembl TrEMBL IDs in Ensembl BioMart Mapping file:", df['UniProtKB/TrEMBL ID'].count())

# Check number of blank Swiss and blank TrEMBL cells in BioMart file. 

print("No. of blank cells in Swiss Prot ID column in Ensembl Biomart file:",
      df['UniProtKB/Swiss-Prot ID'].isna().sum())
print("No. of blank cells in TrEMBL ID column in Ensembl Biomart file:",
      df['UniProtKB/TrEMBL ID'].isna().sum())


No. Ensembl Swiss Prot IDs in Ensembl BioMart Mapping file: 16417
No. Ensembl TrEMBL IDs in Ensembl BioMart Mapping file: 42546
No. of blank cells in Swiss Prot ID column in Ensembl Biomart file: 54092
No. of blank cells in TrEMBL ID column in Ensembl Biomart file: 27963


In [37]:
# Check how many Protein IDs there are in the Hansen Protein MaxQuant file: 

df= pd.read_excel(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\Files used to verify method of integration\proteinGroups.xlsx")

print("No. rows in Majority Protein IDs column in Hansen Protein MaxQuant file:", df['Majority protein IDs'].count())

# Check number of Gene names again, out of curiosity

print("No. Gene names in Hansen Protein MaxQuant file:", df['Gene names'].count())

# In the MaxQuant file, there were blank 'Gene names' cells.
# Check number of blank Majority Protein ID cells in Hansen MaxQuant file.

print("No. of blank cells in 'Majority protein IDs' column in Hansen MaxQuant file:",
      df['Majority protein IDs'].isna().sum())



No. rows in Majority Protein IDs column in Hansen Protein MaxQuant file: 7632
No. Gene names in Hansen Protein MaxQuant file: 7165
No. of blank cells in 'Majority protein IDs' column in Hansen MaxQuant file: 0


In [38]:
# Note on the use of the 'Majority Protein IDs column and not the 'Protein IDs' column in the MaxQuant file: 
# LC- MS/MS identifies peptide fragments, which are extrapolated back to their parent protein. 
# However, many proteins (related or non) may share peptide fragments. 
# MaxQuant groups these ambiguous and indistinguishable peptide fragements into protein groups- which is what a single row in the file represents: a protein group created by sharing of a peptide. 

# The Protein IDs column lists Protein IDs as follows: 
# A peptide 'pep' is detected in the sample, Protein B is present and produced pep. Several other peptides belonging to Protein B have been identified. Therefore, protein B is confidently present in the sample. 
# Protein A also produces pep, however no other peptides of Protein A were detected in the sample. Therefore, the only evidence for the presence of Protein A, is the shared peptide fragment 'pep'. 
# Generally, every peptide fragment produced by Protein A is also all produced by Protein B. 
# Therefore, Protein A is not listed as a separate entry in the MaxQuant file, instead it is listed in the same row as Protein B: Protein B; Protein A. 
# The first accession is the leading protein with the strongest evidence (confirm), however everything thereafter is not ranked and appears in no specific order.  

# The Majority Protein IDs lists Protein IDs as follows: 
# Here, a stricter filter within the same group is applied. 
# The trailing Protein IDs are only retained if it contains a minimum of 50% of the peptide produced by the leading protein. 
# E.g: Protein B is the leading protein with 50 detected peptides. Protein A needs to share at least 25 peptides to remain listed in the group.
# If Protein A has less that 25 shared peptides, this accession is dropped and excluded from this protein group. 
# Weekly supported Protein IDs can belong to a different gene, attaching a peptide count for a protein that was otherwise not present in the sample. 


In [39]:
Hansen = pd.read_excel(
    r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\Files used to verify method of integration\proteinGroups.xlsx",
    usecols=['Gene names', 'Protein IDs', 'Majority protein IDs',
             'Sum_Peptides BAT', 'Sum_Peptides H', 'Sum_Peptides SKM']
)

# Just copied the above code down from previous cells. 


# Under the Protein ID columns, there are 7632 rows.
# However, several of these rows contain multiple Protein IDs, beginning with the leading protein (protein with most peptide evidence) and followed by the trailing proteins in no particular order. 
# I would like to know, how many total Protein IDs are present in the 'Protein IDs' column and in the 'Majority Protein IDs' column, the code below serves that purpose. 
for col in ['Protein IDs', 'Majority protein IDs']:
    lengths = Hansen[col].astype("string").str.split(";").map(len)
    print(f"{col}: {len(Hansen)}, Total IDs: {lengths.sum()}, Largest Protein Group: {lengths.max()}")

# The above code in layman's terms: 
# For the columns 'Protein IDs' and 'Majority Protein IDs'"Preprocessing Algoirthm of Hansen et al., 
# Hansen[col]: fetch columns from that dataframe/file, 
# astype("string"): Pandas will treat each cell as text. Previously, the data type was 'object'.
# .str.spit(";"): within cells, where several entries are separated by a semicolon, split the cell here and turn this cell into a Python list (cleaner option than exploding and resseting the index)
# (Exploding is the better option when there are no associated values, otherwise those values will summed)
# .map(len):  goes down the column and runs the len function on each cell. A two item list will return a value of 2, etc. 
# The f:  everything encased in the {}, gets calculated and printed as a string. 
# {col}: the column name, 
# {len(Hansen)}:  The number of rows in the column
# {lengths.sum()}:  adds up every number in lengths, which is the total number of Protein IDs in that column. 
# {lengths.max()}:  returns the largest instance of items in a single cell/ the largest protein group. 


Protein IDs: 7632, Total IDs: 23338, Largest Protein Group: 36
Majority protein IDs: 7632, Total IDs: 17810, Largest Protein Group: 34


In [40]:
# Map Gene Names First. 
# A left merge on the Maxquant file, because the Ensembl Biomart file is the whole mouse genome, not specific to mitochondria. 
Hansen = pd.read_excel(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\Files used to verify method of integration\proteinGroups.xlsx")
BioMart = pd.read_csv(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\Files used to verify method of integration\mouse ID mapping.tsv", sep="\t")

Hansen = Hansen[['Gene names','Protein IDs','Majority protein IDs', 'Sum_Peptides BAT', 'Sum_Peptides H', 'Sum_Peptides SKM']]
# This code was copied down from the 'Gene mapping' section above, except now we are exclusively looking at the 'Majority protein IDs' column.


Hansen_protein_mapping = Hansen.copy()

# Make a copy of the Hansen table, no alterations to the original 
# Take the "Majority Protein" column and ascertain that each cell is treated as text, it was likely object dtype before, meaning treated as mixed or non-numeric.
# Split each cell where a semicolon appears, turning it into a Python list
# Place the lists back into the "Gene names" column, replacing the object dtype entries that were there previously. 


Hansen_protein_mapping['Majority protein IDs'] = (
    Hansen_protein_mapping["Majority protein IDs"].astype("string").str.split(";")

)

print(Hansen_protein_mapping.shape)
print(Hansen_protein_mapping["Majority protein IDs"].head(3).tolist())
# This code is to make sure that the split worked correctly. The output should show the cell entries separated by commas. 
print("Largest group:", Hansen_protein_mapping["Majority protein IDs"].map(len).max())
# Confirming that this output is the same as above. 


(7632, 6)
[['Q3UTY5', 'Q3UWJ5', 'A2ADP2', 'Q8CDM3', 'A0A067XG46', 'Q9R0X5-2', 'Q9R0X5'], ['B1AU22', 'B1AU23', 'A0A067XG51', 'P70377-3'], ['A0A075B5K0']]
Largest group: 34


In [41]:
### THE DICTIONARY: 

# Again, the dictionary is being made from the Hansen file, because we cannot merge or map on a column of lists. 
# Pandas will only merge cell to cell, and a single cell in the Hansen file containing multiple protein IDs will not map onto A BioMart cell that holds only one Protein ID per cell. 
# Therefore, we need to create this lookup dictionary, consistent with the format of the Gene lookup dictionary made earlier in this file. 
# Again, it is not a smart choice to explode the cells on each item in the cell and reset the index, because this will sum over the same protein group entry. 
# This is referrred to as an accession-level lookup, whilst maintaining the original row structure. 

protein_lookup = {} # curely brackets specifically for creating an empty dictionary. 

for idx, protein_ids in Hansen_protein_mapping['Majority protein IDs'].items():
    if not isinstance(protein_ids, list): 
        continue
    for protein in protein_ids:
        protein = protein.strip() 
        if protein not in protein_lookup:
            protein_lookup[protein] = [idx]
        else:
            protein_lookup[protein].append(idx)

# The above code in layman's terms: 
# items(). moves through the column and records the row number and cell contents. 
# 'isinstance(protein_ids), list:  checks whether the cell is a list, if not, continue to the next row. 
# 'fpr protein in protein_ids': this is the nested for loop. This moves through each cell and each item in each row. 
# The Protein IDs are stripped. They do not need to be converted to lowercase, because protein accessions are always all uppercase. Therefore, this is not a concern ito formatting mismatches. 
# 'if... if not ... else append': if the protein does not yet exist in the dictionary create an entry for it.
#  If the protein is already in the dictionary, append the row number to the list of row numbers already stored under that specific Protein ID (recording multiple occurences of the same Protein ID and recording all the rows that particular Protein ID appears in). 


print("Unique protein IDs in lookup:", len(protein_lookup))

print("Protein IDs that appear in more than one row:", sum(1 for v in protein_lookup.values() if len(v) > 1))


# gene_lookup.values(): all the values in the dictionary (the lists of row numbers)
# for v in: go through them one at a time, calling each one 'v'
# if len(v) > 1: only keep the ones whose list holds more than one row number
# 1: produce the number 1 for every item that passes the test
# sum(...): add up all those 1s, which gives the count

Unique protein IDs in lookup: 17810
Protein IDs that appear in more than one row: 0


In [42]:
# Curiosity check: same lookup built on the full 'Protein IDs' column instead.
# I want to see how many Protein IDs appear more than once and compare to the 'Majority Protein IDs' column. 
# Write a separate dictionary so the mapping dictionary above is not overwritten.

Hansen_protein_mapping["Protein IDs"] = (
    Hansen_protein_mapping["Protein IDs"].astype("string").str.split(";")
)

protein_lookup_all = {}

for idx, protein_ids in Hansen_protein_mapping["Protein IDs"].items():
    if not isinstance(protein_ids, list):
        continue
    for protein in protein_ids:
        protein = protein.strip()
        if protein not in protein_lookup_all:
            protein_lookup_all[protein] = [idx]
        else:
            protein_lookup_all[protein].append(idx)

print("Unique protein IDs in 'Protein IDs' lookup:", len(protein_lookup_all))
print("Protein IDs that appear in more than one row:",
      sum(1 for v in protein_lookup_all.values() if len(v) > 1))

Unique protein IDs in 'Protein IDs' lookup: 23338
Protein IDs that appear in more than one row: 0


In [43]:
# Brief Summary: 
# Gene lookup:      6765 Unique Gene names, 292 of these Genes appearing in more than one row. 
# Protein ID lookup: 17810 Unique Protein IDs, 0 of these Protein IDs appearing in more than one row. 

# Recalling that: 
# Several distinct proteins can be encoded by the same gene, often these are isoforms and paralogs, descending from the same parent gene. 
# Therefore one gene name can map to several peptide counts in the Hansen file. 
# However, this does not necessarily simplify the protein mapping, because one cell can contain multiple accessions. 
# Therefore, mapping both routes is the safest optio. Where gene name mapping and protein ID mapping agree, the assignment can be taken with more confidence than a single identifier mapping.

# For the first mapping attempt between the Hansen and BioMart file, I will retain the isoform suffixes. 
# Thereafter the suffixes will be stripped and mapping will be done again. 
# This will then be compared to determine whether the suffixes are a hindrance, or if we observe the one-to-many mapping phenomenon only when the suffixes are dropped. 
# This will confirm that the one-to-many mapping trend observed in the gene name mappping is a result of isoforms and paralogs. 



In [44]:
BioMart_protein_mapping = BioMart.copy()
BioMart_protein_mapping['SwissProt_key'] = BioMart_protein_mapping['UniProtKB/Swiss-Prot ID'].str.strip()
BioMart_protein_mapping['TrEMBL_key'] =BioMart_protein_mapping['UniProtKB/TrEMBL ID'].str.strip()

# Recall that these key columns are cleaned copies of the BioMart identifier columns, for matching against the protein lookup dictionary. 

matched_protein_ids = []

# In the Gene name lookup, the code was as follows:
# for BioMart_idx, gene_name in BioMart_gene_mapping["gene_name"].items():
# However, this will not work here because .items() moves through a single column and returns the value in each cell directly. 
# .iterrows() moves through the entire dataset row by row, so both columns within a row can be interacted with. 
#  For this reason, we specify that iterrows must move through the columns named in the above code. 

for BioMart_idx, row in BioMart_protein_mapping.iterrows():
    SwissProt_key = row['SwissProt_key']
    TrEMBL_key = row['TrEMBL_key']



    if pd.notna (TrEMBL_key) and TrEMBL_key in protein_lookup:
        for Hansen_idx in protein_lookup[TrEMBL_key]:
            matched_protein_ids.append({"BioMart_idx": BioMart_idx,
                                    "Hansen_idx": Hansen_idx,
                                    "Matched on": "TrEMBL ID",
                                    "Protein accession": TrEMBL_key})
            
    # Is there an entry in the TrEMBL cell? If yes, is this accession of the ones stored in the previously created lookup dictionary? 
    # Which row number (index) was the TrEMBL ID found in the Hansen file
    # Record the match entry in the matched_protein_ids list, along with the BioMart row number, the Hansen row it points to and the specific identifier column match. 


    if pd.notna(SwissProt_key) and SwissProt_key in protein_lookup:
        for Hansen_idx in protein_lookup[SwissProt_key]:
            matched_protein_ids.append({"BioMart_idx": BioMart_idx,
                                    "Hansen_idx": Hansen_idx,
                                    "Matched on": "Swiss-Prot ID",
                                    "Protein accession": SwissProt_key})



matched_protein_ids_df = pd.DataFrame(matched_protein_ids)

print("Total match pairs:", len(matched_protein_ids_df))
print("Distinct Hansen rows matched:", matched_protein_ids_df["Hansen_idx"].nunique())
print("Distinct BioMart rows matched:", matched_protein_ids_df["BioMart_idx"].nunique())
print(matched_protein_ids_df["Matched on"].value_counts())



Total match pairs: 13932
Distinct Hansen rows matched: 7185
Distinct BioMart rows matched: 13873
Matched on
TrEMBL ID        7307
Swiss-Prot ID    6625
Name: count, dtype: int64


In [45]:
# The number of match events is worth noting. 
# A single Hansen Protein ID cell can contain up to 34 accessions. 
# Each accession can map to a distinct BioMart row.
# Therefore a single cell in the protein group row in the Hansen file can generate many match events, each with a single Protein ID in the BioMart file. 

# Also, different and separate genes at different positions in the genome can produce identical proteins. 
# E.g: The H4 protein (mouse): Histones are required in large quantities, because once a cell replicates its genome, the DNA must be packaged immediately.
# Each nucleosome requires two copies of H4. The genome therefore carries about a dozen H4 genes, transcribed together during S phase:
# parallel capacity instead of making one gene work overtime. 
# The DNA sequences of these copies are not identical, but they differ only at silent positions where a base change does not alter the amino acid. H4 is also under severe structural constraint
#  since it must fit against H3, H2A, H2B and the DNA backbone, so the protein sequence itself has stayed the same.

# Consequence for the mapping: 
# UniProt catalogues proteins and assigns one entry (P62806)
# Ensembl catalogues genes, so the thirteen H4 loci each have their own gene ID. 
# BioMart maps this by listing the 13 different genes, but with the same protein ID at each one.  Therefore, a single Protein ID can appear in multiple rows in the BioMart file. 
# Therefore, a single Hansen Protein ID in a Hansen cell can match to 13 different genes in the BioMart file, all claiming the same peptide count.
# The peptides are identical whichever gene was transcribed, so the experiment cannot distinguish them.
# This ambiguity is real rather than a mapping defect.



#                     ┌─ P62806 ─┬─→ H4c2
#                     │          ├─→ H4c11
# Hansen row ─────────┤          └─→ H4c12
# (1 peptide count)   ├─ Q3UTY5 ──→ Nnt
#                     └─ A2ADP2 ──→ Ogdh

In [46]:
# Take matched_protein_ids_df (the pairs table holding BioMart_idx and Hansen_idx) and merge new columns onto it
# Fetch those columns from BioMart_gene_mapping, taking only Gene stable ID, Gene name, UniProtKB/Swiss-Prot ID and UniProtKB/TrEMBL ID
# Use the number in matched_protein_ids_df BioMart_idx column as a row position in BioMart_gene_mapping
# Keep every pair already in matched_protein_ids_df, even if no BioMart row is found

matched_protein_ids_df = matched_protein_ids_df.merge(
    BioMart_protein_mapping[["UniProtKB/Swiss-Prot ID",  "UniProtKB/TrEMBL ID", "Gene stable ID", "Gene name"]],
    left_on = "BioMart_idx",
    right_index=True,
    how="left"                           
)

print(matched_protein_ids_df.shape)
print(matched_protein_ids_df.head(20))


(13932, 8)
    BioMart_idx  Hansen_idx     Matched on Protein accession  \
0             0        2685  Swiss-Prot ID            P03888   
1             1        2686  Swiss-Prot ID            P03893   
2             2        2656  Swiss-Prot ID            P00397   
3             3        2657  Swiss-Prot ID            P00405   
4             4        2690  Swiss-Prot ID            P03930   
5             5        2665  Swiss-Prot ID            P00848   
6             6        2658  Swiss-Prot ID            P00416   
7             7        2687  Swiss-Prot ID            P03899   
8             9        2688  Swiss-Prot ID            P03911   
9            10        2689  Swiss-Prot ID            P03921   
10           12        2650  Swiss-Prot ID            P00158   
11           93        3801  Swiss-Prot ID            Q05738   
12          185        4435  Swiss-Prot ID            Q62095   
13          228        7272  Swiss-Prot ID            Q9Z0N2   
14          373        6155  

In [47]:
# This is the second merge, because after created the dictionary, the stored pairs table only hold the row number corresponding to where the matches originate from in each file.
# The first merge used the created BioMart_idx to fetch the BioMart details for that particular row. 
# The second merge uses the Hansen_idx to fetch the correpspnding Hansen details in the indexed row. 
# Merges on both sides are needed to map the matched indeces back to the rest of the information in their corresponding rows.

matched_protein_ids_df = matched_protein_ids_df.merge(
    Hansen[["Gene names", "Majority protein IDs", "Sum_Peptides BAT", "Sum_Peptides H", "Sum_Peptides SKM"]],
    left_on="Hansen_idx",
    right_index=True,
    how="left"                           
)

print(matched_protein_ids_df.shape)
print(matched_protein_ids_df.head(20))


(13932, 13)
    BioMart_idx  Hansen_idx     Matched on Protein accession  \
0             0        2685  Swiss-Prot ID            P03888   
1             1        2686  Swiss-Prot ID            P03893   
2             2        2656  Swiss-Prot ID            P00397   
3             3        2657  Swiss-Prot ID            P00405   
4             4        2690  Swiss-Prot ID            P03930   
5             5        2665  Swiss-Prot ID            P00848   
6             6        2658  Swiss-Prot ID            P00416   
7             7        2687  Swiss-Prot ID            P03899   
8             9        2688  Swiss-Prot ID            P03911   
9            10        2689  Swiss-Prot ID            P03921   
10           12        2650  Swiss-Prot ID            P00158   
11           93        3801  Swiss-Prot ID            Q05738   
12          185        4435  Swiss-Prot ID            Q62095   
13          228        7272  Swiss-Prot ID            Q9Z0N2   
14          373        6155 

In [48]:
# The above output contains only the matched rows. 
# In this merge step, Hansen is on the left making it the reference dataset. 
# All the Hansen rows whose accessions found no match in the BioMart file are included at this step, which will show as blanks in the BioMart columns.

Hansen_BioMart_ProteinIDs = Hansen.merge(
    matched_protein_ids_df[["BioMart_idx", "Hansen_idx", "Matched on", "Protein accession",
                            "Gene stable ID", "Gene name",
                            "UniProtKB/Swiss-Prot ID", "UniProtKB/TrEMBL ID"]],
    left_index=True,
    right_on="Hansen_idx",
    how="left"
)

#Unmatched rows have no BioMart row, so BioMart_idx is blank there, which forces decimals (93.0).
#"Int64" (capital I) keeps whole numbers while allowing blanks.
Hansen_BioMart_ProteinIDs["BioMart_idx"] = Hansen_BioMart_ProteinIDs["BioMart_idx"].astype("Int64")

print(Hansen_BioMart_ProteinIDs.shape)

(14379, 14)


In [49]:
print(Hansen_BioMart_ProteinIDs["Matched on"].head())

4037.0    Swiss-Prot ID
4041.0        TrEMBL ID
4044.0        TrEMBL ID
4045.0    Swiss-Prot ID
4046.0        TrEMBL ID
Name: Matched on, dtype: object


In [50]:
# Similarly to the Gene names section, I want to record the source of the match in the final excel files.
Prot_Identifier = Hansen_BioMart_ProteinIDs.groupby("Hansen_idx")["Matched on"].apply(set)
#  Group together all records belonging to the same Hansen row.
# 

protein_id_source = []


for index, row in Hansen_BioMart_ProteinIDs.iterrows():
    Hansen_idx = row["Hansen_idx"]

    if pd.isna(row["Majority protein IDs"]):
        protein_id_source.append("No Protein ID in Hansen MaxQuant")
    elif pd.isna(row["Matched on"]):
        protein_id_source.append("Hansen MaxQuant Only")
    elif Prot_Identifier[Hansen_idx] == {"Swiss-Prot ID", "TrEMBL ID"}:
        protein_id_source.append("Hansen MaxQuant, BioMart (Swiss-Prot and TrEMBL)")
    elif Prot_Identifier[Hansen_idx] == {"Swiss-Prot ID"}:
        protein_id_source.append("Hansen MaxQuant, BioMart (Swiss-Prot)")
    elif Prot_Identifier[Hansen_idx] == {"TrEMBL ID"}:
        protein_id_source.append("Hansen MaxQuant, BioMart (TrEMBL)")


Hansen_BioMart_ProteinIDs["Protein ID source"] = protein_id_source


# if pd.isna(row["Majority protein IDs"]): is this row's cell empty?
# elif pd.isna(row["Matched on"]): This is where no match for the Hansen Protein ID was found in the BioMart file
# elif row["Matched on"]== "Swiss-Prot ID": A match between the Hansen and BioMart files exist, arrising from the same Protein ID in the Swiss-Prot column
# elif row["Matched on"] == "TrEMBL ID": A match between the Hansen and BioMart files exist, arrising from the same Protein ID in the TrEMBL column

print(len(protein_id_source), len(Hansen_BioMart_ProteinIDs))
print(Hansen_BioMart_ProteinIDs["Protein ID source"].value_counts())
   

14379 14379
Protein ID source
Hansen MaxQuant, BioMart (Swiss-Prot and TrEMBL)    8211
Hansen MaxQuant, BioMart (Swiss-Prot)               3458
Hansen MaxQuant, BioMart (TrEMBL)                   2263
Hansen MaxQuant Only                                 447
Name: count, dtype: int64


In [51]:
pd.set_option("display.max_rows", 500)

# What do the Protein IDs look like for the Hansen MaxQuant proteins that were not matched to any accession in the BioMart file.

print(Hansen_BioMart_ProteinIDs[
    Hansen_BioMart_ProteinIDs["Protein ID source"] == "Hansen MaxQuant Only"
]["Majority protein IDs"].head(450))

NaN                                A0A087WQ00;A0A087WNV3
NaN                                           A0A087WQH8
NaN                                           A0A571BEE9
NaN                                           A0A5F8MPX4
NaN                                             A2AB59-2
NaN                                             A2ALU4-2
NaN                                             A2ASS6-3
NaN                                             A6H6A9-3
NaN                                               B0F2B4
NaN                                               B1ATV3
NaN                                               B1AXF3
NaN          CON__ENSEMBL:ENSBTAP00000007350;CON__P01030
NaN                      CON__ENSEMBL:ENSBTAP00000031900
NaN                      CON__ENSEMBL:ENSBTAP00000038253
NaN           CON__ENSEMBL:ENSP00000377550;CON__P13646-1
NaN                                          CON__P00761
NaN                                          CON__P01966
NaN                            

In [52]:
# Some of the entries in the MaxQuant file are contaminants. These are real proteins, but from the wrong source. Possibly keratinm trypsin, bonine albumin, etc. 

print("Number of Hansen MaxQuant Protein IDs that are Contaminants:", Hansen_BioMart_ProteinIDs[Hansen_BioMart_ProteinIDs["Protein ID source"] == "Hansen MaxQuant Only"
]["Majority protein IDs"].str.startswith("CON__").sum())



print("Number of Hansen MaxQuant Protein IDs that are Reverse Sequence Decoys:", Hansen_BioMart_ProteinIDs[Hansen_BioMart_ProteinIDs["Protein ID source"] == "Hansen MaxQuant Only"
]["Majority protein IDs"].str.startswith("REV__").sum())

# In Maxquant, REV marks a decoy. This is how the false discovery rate is determined. 
# Each real sequence (From the FASTA file MaxQuant begins with) is reversed: same amino acids, same length, but biologically irrelevant. 
# The search to match the peptides outputted from LC-MS/MS to the observed spectrum of sequences in the FASTA file is ran, cosnidering both the actual protein sequence and the decoy. 
# Any spectrum that matches the reversed sequenced is considered false. These are counted to arrive at the error rate. 
# If 1% of the hits/matches are decoys, then it is assumed that approximately 1% of the confirmed hits may be false as well. 

Number of Hansen MaxQuant Protein IDs that are Contaminants: 35
Number of Hansen MaxQuant Protein IDs that are Reverse Sequence Decoys: 266


In [53]:
# This means that out of the 447 non-matches, only 146 need to be mapped.
# The contaminants and reverse sequence decoys can be discounted. 
# Hoping to see that some of the 146 were mapped via Gene name. 

In [54]:
print("Merged table rows:", len(Hansen_BioMart_ProteinIDs))


print("Hansen rows matched:", Hansen_BioMart_ProteinIDs.loc[Hansen_BioMart_ProteinIDs["Matched on"].notna(), "Hansen_idx"].nunique())
# In the merged Hansen_BioMart_ProteinIds file, count the number of uniqiue (nununique) Hansen rows that have a protein ID match in BioMart.


Merged table rows: 14379
Hansen rows matched: 7185


In [55]:
# For each Hansen row, record which identifier the Hansen Protein ID mapped to in the BioMart file - Swiss-Prot or TrEMBL.
# groupby(Hansen_idx) groups rows together based on a particular column, this returns the Hansen index and what the identifier it matched on:
# For this particular group, only consider the "matched on" column. 
# apply(): perform this function on each group (a group is one Hansen row)
# apply(set): for each Hansen row, collects the unique types of matches it had, either Swiss-Prot ID and TrEMBL ID, Swiss-Prot only or TrEMBL only.
Prot_Identifier = Hansen_BioMart_ProteinIDs.groupby("Hansen_idx")["Matched on"].apply(set)


print("Matched on both:", (Prot_Identifier.map(len) == 2).sum())
# For each Hansen row, count the number of distinct lables collected. For those that collected two, sum those over. 
# The output will therefore be indicative of how many IDs mapped on both the Swiss and TrEMBL ID. 
# '==2' is a way of saying 'both'. 

print("Swiss-Prot only:", (Prot_Identifier == {"Swiss-Prot ID"}).sum())
# How many IDs between the Hansen and the BioMart file matched on Swiss-Prot IDs
print("TrEMBL only:", (Prot_Identifier == {"TrEMBL ID"}).sum())
# How many IDs between the Hansen and the BioMart file matched on TrEMBL IDs

Matched on both: 2801
Swiss-Prot only: 3272
TrEMBL only: 1112


In [56]:
# This code is to count how many BioMart rows were present, but mapped to no Protein ID in the Hansen file.
# unique(): strips any repeats where a BioMart row was hit more than once by a Hansen Protein ID and  returns the distinct values as an array, not a count of how many are unique (contrast to nununique).
# isin(): Is each BioMart row number in the array? is it one of the values in the distinct array. 
# BUT "~" inverts the answer (new to me). Therefore, this code checks for BioMart Protein IDs that do not at all appear in the distinct array. 



matched_bm_rows = matched_protein_ids_df["BioMart_idx"].unique()
BioMart_only = BioMart_protein_mapping[~BioMart_protein_mapping.index.isin(matched_bm_rows)]
print("BioMart rows with no match:", len(BioMart_only))

# Therefore this code tells me how many BioMart rows are not in the matched, distinct array of row numbers.
# Note this counts rows, not Protein IDs: BioMart holds one row per gene-identifier combination, so a single accession can occupy several rows.


unmatched_accessions = set(BioMart_only["UniProtKB/Swiss-Prot ID"].dropna()) | set(BioMart_only["UniProtKB/TrEMBL ID"].dropna())
print("Distinct accessions with no match:", len(unmatched_accessions))

BioMart rows with no match: 56636
Distinct accessions with no match: 42182


In [57]:
unmatched = Hansen_BioMart_ProteinIDs[
    Hansen_BioMart_ProteinIDs["Protein ID source"] == "Hansen MaxQuant Only"
]

contaminants = unmatched[unmatched["Majority protein IDs"].str.startswith("CON__")]
decoys = unmatched[unmatched["Majority protein IDs"].str.startswith("REV__")]
real_unmatched = unmatched[~unmatched["Majority protein IDs"].str.startswith(("CON__", "REV__"))]

print(len(contaminants), len(decoys), len(real_unmatched))

35 266 146


In [58]:
# The 146 remaining unmatched rows fail for one of two reasons. Split them so the genuine failures are
# separated from the ones failing only on the isoform suffix.

all_biomart_accessions = set(BioMart["UniProtKB/Swiss-Prot ID"].dropna()) | set(BioMart["UniProtKB/TrEMBL ID"].dropna())

def base_form_in_biomart(cell):
    return any(a.strip().split("-")[0] in all_biomart_accessions for a in str(cell).split(";"))

isoform_failures = real_unmatched[real_unmatched["Majority protein IDs"].map(base_form_in_biomart)]
absent_from_biomart = real_unmatched[~real_unmatched["Majority protein IDs"].map(base_form_in_biomart)]

print(len(isoform_failures), len(absent_from_biomart))

106 40


In [59]:
Hansen_BioMart_ProteinIDs[[
    "BioMart_idx",
    "Hansen_idx",
    "Gene stable ID",
    "Majority protein IDs",
    "Protein accession",
    "Protein ID source"
]].head(10)

,BioMart_idx,Hansen_idx,Gene stable ID,Majority protein IDs,Protein accession,Protein ID source
4037.0,23972,0,ENSMUSG00000031174,Q3UTY5;Q3UWJ5;A2ADP2;Q8CDM3;A0A067XG46;Q9R0X5-...,Q9R0X5,"Hansen MaxQuant, BioMart (Swiss-Prot and TrEMBL)"
4041.0,23989,0,ENSMUSG00000031174,Q3UTY5;Q3UWJ5;A2ADP2;Q8CDM3;A0A067XG46;Q9R0X5-...,Q3UTY5,"Hansen MaxQuant, BioMart (Swiss-Prot and TrEMBL)"
4044.0,23993,0,ENSMUSG00000031174,Q3UTY5;Q3UWJ5;A2ADP2;Q8CDM3;A0A067XG46;Q9R0X5-...,A0A067XG46,"Hansen MaxQuant, BioMart (Swiss-Prot and TrEMBL)"
4045.0,23993,0,ENSMUSG00000031174,Q3UTY5;Q3UWJ5;A2ADP2;Q8CDM3;A0A067XG46;Q9R0X5-...,Q9R0X5,"Hansen MaxQuant, BioMart (Swiss-Prot and TrEMBL)"
4046.0,23994,0,ENSMUSG00000031174,Q3UTY5;Q3UWJ5;A2ADP2;Q8CDM3;A0A067XG46;Q9R0X5-...,A2ADP2,"Hansen MaxQuant, BioMart (Swiss-Prot and TrEMBL)"
4047.0,23994,0,ENSMUSG00000031174,Q3UTY5;Q3UWJ5;A2ADP2;Q8CDM3;A0A067XG46;Q9R0X5-...,Q9R0X5,"Hansen MaxQuant, BioMart (Swiss-Prot and TrEMBL)"
4683.0,27203,1,ENSMUSG00000031137,B1AU22;B1AU23;A0A067XG51;P70377-3,A0A067XG51,"Hansen MaxQuant, BioMart (TrEMBL)"
4684.0,27205,1,ENSMUSG00000031137,B1AU22;B1AU23;A0A067XG51;P70377-3,B1AU22,"Hansen MaxQuant, BioMart (TrEMBL)"
4685.0,27207,1,ENSMUSG00000031137,B1AU22;B1AU23;A0A067XG51;P70377-3,B1AU23,"Hansen MaxQuant, BioMart (TrEMBL)"
267.0,2099,2,ENSMUSG00000094345,A0A075B5K0,A0A075B5K0,"Hansen MaxQuant, BioMart (TrEMBL)"


In [60]:
# Create the source columns to be displayed in the excel file:

protein_source_cols = ["BioMart_idx", "Hansen_idx", "Gene stable ID", "Majority protein IDs", "Protein accession", "Protein ID source"]

# Sheet 1: Every Hansen row under the specific columns. 
Hansen_BioMart_ProteinIDs_audit = Hansen_BioMart_ProteinIDs[protein_source_cols]

# Sheet 2: This contains only the rows that with a paired match between both the Hansen and the BioMart file. 
# .isin() rather than '==', because there are two identifiers we need to consider.
matched_protein_audit = Hansen_BioMart_ProteinIDs[
    Hansen_BioMart_ProteinIDs["Protein ID source"].isin([
        "Hansen MaxQuant, BioMart (Swiss-Prot and TrEMBL)",
        "Hansen MaxQuant, BioMart (Swiss-Prot)",
        "Hansen MaxQuant, BioMart (TrEMBL)"])][protein_source_cols]

maxquant_only_protein_audit = Hansen_BioMart_ProteinIDs[
    Hansen_BioMart_ProteinIDs["Protein ID source"] == "Hansen MaxQuant Only"][protein_source_cols]


# Sheets 4 to 7 split those 447 by the reason they failed, so the genuine failures can be separated
# from the ones that should not have matched in the first place.

# Sheet 4: contaminants. Real proteins from the wrong source (keratin, trypsin, bovine albumin).
# These have no mouse Ensembl gene, so failing to match is correct.
contaminants_audit = contaminants[protein_source_cols]

# Sheet 5: reverse sequence decoys. Fabricated sequences MaxQuant generates to estimate the false
# discovery rate. These do not exist, so failing to match is correct.
decoys_audit = decoys[protein_source_cols]

# Sheet 6: rows failing only because of the isoform suffix. The base accession is present in BioMart,
# but the suffixed form is not, since BioMart contains no isoform-suffixed accessions.
isoform_audit = isoform_failures[protein_source_cols]

# Sheet 7: real accessions genuinely absent from the BioMart export. These are the only true mapping
# failures, and are candidates for rescue via the gene name route.
absent_audit = absent_from_biomart[protein_source_cols]



with pd.ExcelWriter(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\protein_id_mapping_Hansen_BioMart.xlsx") as writer:
    Hansen_BioMart_ProteinIDs_audit.to_excel(writer, sheet_name="All Hansen rows", index=False)
    matched_protein_audit.to_excel(writer, sheet_name="Matched Hansen and BioMart", index=False)
    maxquant_only_protein_audit.to_excel(writer, sheet_name="Hansen MaxQuant only", index=False)
    contaminants_audit.to_excel(writer, sheet_name="Contaminants", index=False)
    decoys_audit.to_excel(writer, sheet_name="Reverse decoys", index=False)
    isoform_audit.to_excel(writer, sheet_name="Isoform suffix failures", index=False)
    absent_audit.to_excel(writer, sheet_name="Absent from BioMart", index=False)


_________________________________________________________________________________________________

_________________________________________________________________________________________________

# Joining Gene and Protein Mapping Files

In [61]:
# I am aware that the same Ensembl ID can correspond to multiple Protein IDs, which will each have a different peptide count attached to it. 
# For this final mapping file, I will keep the Hansen_idx as an output column, so that I can visibly see that these instances arise from separate rows in the Hansen file.
# Without Hansen_idx, I would only see the same Ensembl ID and gene name associated with different protein IDs and peptide counts. 
# Retaining the index provides traceability and allows these multiple protein-to-gene mappings to be linked back to their original Hansen rows.

In [62]:
# Gene Names

# The drop duplicates here will not effect the multiple protein-to-gene mappings, because it will only drop duplicates of an entire row of listen columns (hence Hansen_idx).  

gene_mapping_assignments = Hansen_BioMart_GeneNames.loc[
    Hansen_BioMart_GeneNames["Gene stable ID"].notna(),
    ["Hansen_idx", "Gene stable ID", "Gene names", "Gene name", "Gene name source"]
].drop_duplicates()

gene_mapping_assignments["Gene name match"] = True


print("Distinct gene name mapping assignments:", len(gene_mapping_assignments))
print(gene_mapping_assignments.head(20))


Distinct gene name mapping assignments: 6927
        Hansen_idx      Gene stable ID                             Gene names  \
7645.0           0  ENSMUSG00000031174                                   Rpgr   
9209.0           1  ENSMUSG00000031137                                  Fgf13   
250.0            2  ENSMUSG00000094345                             Igkv14-126   
251.0            3  ENSMUSG00000096632                              Igkv9-124   
256.0            4  ENSMUSG00000093861  Igkv1-110;Igkv1-35;Igkv1-99;Igkv1-115   
537.0            4  ENSMUSG00000076525  Igkv1-110;Igkv1-35;Igkv1-99;Igkv1-115   
543.0            4  ENSMUSG00000076573  Igkv1-110;Igkv1-35;Igkv1-99;Igkv1-115   
541.0            5  ENSMUSG00000076534                              Igkv12-89   
542.0            6  ENSMUSG00000076536                               Igkv4-86   
507.0            7  ENSMUSG00000076550                               Igkv4-63   
572.0            8  ENSMUSG00000076569                          

In [63]:
# Protein IDs


# matched_protein_ids_df contains 13 932 rows. 
# Recall that a match to both Swiss-Prot or TrEMBL was initially recorded as two separate entries.
# I would like this to become one row, so that for a Hansen row that mapped on both, the "Matched on" cell reads {"Swiss-Prot ID", "TrEMBL ID"}.

# groupby(["Hansen_idx", "Gene stable ID"]) bundles together every match sharing the same Hansen row and the same Ensembl gene. That pair is the assignment, and it is what the two tables will be joined on.

# .agg means aggregate. This function tells Python to combine several rows into one. 
# Where a Hansen row matched the same Ensembl gene on both accessions from the BioMart file, Python needs to know what should appear in the single aggregated row.
# "first": takes the first value in the Majority protein IDs column, which for these entries will be the same and the same principle is applied to the UniProtKB/Swiss-Prot ID and UniProtKB/TrEMBL ID columns.

# .reset_index(): after groupby, the two grouping columns become the table's index rather than ordinary columns. reset_index() pushes them back out into columns, since the merge below needs them as columns.


prot_mapping_assignments = matched_protein_ids_df.groupby(["Hansen_idx", "Gene stable ID"]).agg({
    "Matched on": set,
    "Majority protein IDs": "first",
    "UniProtKB/Swiss-Prot ID": "first",
    "UniProtKB/TrEMBL ID": "first"
}).reset_index()


# lambda s: a function where 's' is the current cell or the set. 
# Swiss-Prot in s, checks whether this ID is in that set and returns True if yes, False if not and same applies for the TrEMBL IDs. 

prot_mapping_assignments["Swiss-Prot match"] = prot_mapping_assignments["Matched on"].map(lambda s: "Swiss-Prot ID" in s)
prot_mapping_assignments["TrEMBL match"] = prot_mapping_assignments["Matched on"].map(lambda s: "TrEMBL ID" in s)

print("Distinct protein ID assignments:", len(prot_mapping_assignments))
print(prot_mapping_assignments.head(20))


Distinct protein ID assignments: 7514
    Hansen_idx      Gene stable ID                  Matched on  \
0            0  ENSMUSG00000031174  {Swiss-Prot ID, TrEMBL ID}   
1            1  ENSMUSG00000031137                 {TrEMBL ID}   
2            2  ENSMUSG00000094345                 {TrEMBL ID}   
3            3  ENSMUSG00000096632                 {TrEMBL ID}   
4            4  ENSMUSG00000076525                 {TrEMBL ID}   
5            4  ENSMUSG00000076535                 {TrEMBL ID}   
6            4  ENSMUSG00000076573                 {TrEMBL ID}   
7            4  ENSMUSG00000093861                 {TrEMBL ID}   
8            4  ENSMUSG00000094335                 {TrEMBL ID}   
9            4  ENSMUSG00000095497                 {TrEMBL ID}   
10           5  ENSMUSG00000076534                 {TrEMBL ID}   
11           6  ENSMUSG00000076536                 {TrEMBL ID}   
12           7  ENSMUSG00000076550                 {TrEMBL ID}   
13           8  ENSMUSG00000076569    

In [64]:
# Join the two mapping tables on the assignment: the pair of Hansen row and Ensembl gene.
# on=["Hansen_idx", "Gene stable ID"] matches on both columns together. Matching on Gene stable ID alone
# would pair every gene-route row for a gene against every protein-route row for it, producing a table
# of millions of rows.
# how="outer" keeps rows found on either side, unlike the how="left" merges used earlier which keep only
# the left table's rows. An assignment made only by the gene route survives, and so does one made only by
# the protein route. This is what makes the rescues in either direction visible.
combined_mapping = gene_mapping_assignments.merge(
    prot_mapping_assignments,
    on=["Hansen_idx", "Gene stable ID"],
    how="outer"
)

# Rows present on only one side have blanks in the other side's flag columns. fillna(False) turns those
# into False, so every row carries a usable True/False answer for all three identifiers.
for col in ["Gene name match", "Swiss-Prot match", "TrEMBL match"]:
    combined_mapping[col] = combined_mapping[col].fillna(False)

print("Combined assignments:", len(combined_mapping))
print("Distinct Hansen rows:", combined_mapping["Hansen_idx"].nunique())
print("Distinct Ensembl genes:", combined_mapping["Gene stable ID"].nunique())

Combined assignments: 7659
Distinct Hansen rows: 7302
Distinct Ensembl genes: 7245


In [65]:
print(combined_mapping.groupby(["Gene name match", "Swiss-Prot match", "TrEMBL match"]).size())

Gene name match  Swiss-Prot match  TrEMBL match
False            False             True             244
                 True              False            337
                                   True             151
True             False             False            145
                                   True            1019
                 True              False           3133
                                   True            2630
dtype: int64


In [66]:
# Build the combined source column from the three flags. Unlike the source columns in each section, this
# one is assembled rather than chosen: each identifier that matched contributes its name to the label.
all_identifiers = []

for index, row in combined_mapping.iterrows():
    identifiers = []
    if row["Gene name match"]:
        identifiers.append("Gene name")
    if row["Swiss-Prot match"]:
        identifiers.append("Swiss-Prot")
    if row["TrEMBL match"]:
        identifiers.append("TrEMBL")
    all_identifiers.append(", ".join(identifiers))

combined_mapping["All identifiers"] = all_identifiers

print(combined_mapping["All identifiers"].value_counts())

All identifiers
Gene name, Swiss-Prot            3133
Gene name, Swiss-Prot, TrEMBL    2630
Gene name, TrEMBL                1019
Swiss-Prot                        337
TrEMBL                            244
Swiss-Prot, TrEMBL                151
Gene name                         145
Name: count, dtype: int64


In [67]:
intermediate_cols =["Hansen_idx", "Gene stable ID", "Gene names", "Gene name", "Gene name source",
                     "Majority protein IDs", "UniProtKB/Swiss-Prot ID", "UniProtKB/TrEMBL ID",
                     "All identifiers"]

with pd.ExcelWriter(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\intermediate_gene_protein_mapping.xlsx") as writer:
    combined_mapping[intermediate_cols].to_excel(writer, sheet_name="Combined mapping", index=False)

In [68]:
# 7659 matches between the Hansen and the BioMart files on one or more identifiers.


# The above file contains the confirmed matches only. 
# Before mapping onto MitoMammal file, I need to merge back the 330 Hansen rows that did not map to anything in the BioMart file. 
# Also need to add on the relative protein abundance columns from MitoMammla and the Sum Peptide columns I made in the Hansen MaxQuant file. 

In [69]:
# Merge the combined mapping back onto the full Hansen table.
# Hansen is the left table with how="left", so all 7632 rows are kept: the 330 that matched by neither
# route return here with blanks in every mapping column.
# The peptide count columns come across from Hansen at the same time.
merge_back_mapping = Hansen.merge(
    combined_mapping.drop(columns=["Gene names", "Majority protein IDs"]),
    left_index=True,
    right_on="Hansen_idx",
    how="left"
)

print("Rows:", len(merge_back_mapping))
print("Hansen rows with no assignment:", merge_back_mapping["Gene stable ID"].isna().sum())

Rows: 7989
Hansen rows with no assignment: 330


In [70]:
merge_back_mapping["All identifiers"] = merge_back_mapping["All identifiers"].fillna(
    "No match on Gene name, Swiss-Prot or TrEMBL")

In [71]:
print(merge_back_mapping.columns.tolist())

['Gene names', 'Protein IDs', 'Majority protein IDs', 'Sum_Peptides BAT', 'Sum_Peptides H', 'Sum_Peptides SKM', 'Hansen_idx', 'Gene stable ID', 'Gene name', 'Gene name source', 'Gene name match', 'Matched on', 'UniProtKB/Swiss-Prot ID', 'UniProtKB/TrEMBL ID', 'Swiss-Prot match', 'TrEMBL match', 'All identifiers']


In [72]:
# Load the MitoMammal reference files. Neither has a header row, so names are supplied.
MitoMammal_BAT = pd.read_csv(
    r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\Files used to verify method of integration\BAT_data.tsv",
    sep="\t", header=None, names=["Gene stable ID", "Sum Peptide Counts MitoMammal BAT"])

MitoMammal_Heart = pd.read_csv(
    r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\Files used to verify method of integration\Heart_data.tsv",
    sep="\t", header=None, names=["Gene stable ID", "Sum Peptide Counts MitoMammal Heart"])

# Attach the MitoMammal abundance values, keyed on the Ensembl gene ID.
with_mitomammal = merge_back_mapping.merge(MitoMammal_BAT, on="Gene stable ID", how="left")
with_mitomammal = with_mitomammal.merge(MitoMammal_Heart, on="Gene stable ID", how="left")

with_mitomammal["In MitoMammal"] = with_mitomammal["Gene stable ID"].isin(
    set(MitoMammal_BAT["Gene stable ID"]))

merge_cols = [
    "Hansen_idx",
    "Gene stable ID",
    "In MitoMammal",
    "Gene names",
    "Gene name",
    "Gene name source",
    "Majority protein IDs",
    "UniProtKB/Swiss-Prot ID",
    "UniProtKB/TrEMBL ID",
    "All identifiers",
    "Sum_Peptides H",
    "Sum Peptide Counts MitoMammal Heart",
    "Sum_Peptides BAT",
    "Sum Peptide Counts MitoMammal BAT",
    "Sum_Peptides SKM",
]

full_mapping_output = with_mitomammal[merge_cols]

# Trimmed to only the Ensembl IDs MitoMammal uses, with the MitoMammal ID first.
mitomammal_mapping = full_mapping_output[full_mapping_output["In MitoMammal"]].drop(columns="In MitoMammal")
mitomammal_mapping.insert(0, "MitoMammal Ensembl ID", mitomammal_mapping["Gene stable ID"])
mitomammal_mapping = mitomammal_mapping.rename(columns={"Gene stable ID": "BioMart Ensembl ID"})



In [73]:
# Hansen rows that reached an Ensembl gene, with every column kept
mapping_for_spine = merge_back_mapping.dropna(subset=["Gene stable ID"]).copy()
mapping_for_spine["Hansen_idx"] = mapping_for_spine["Hansen_idx"].astype(int)
mapping_for_spine["Matched on"] = mapping_for_spine["Matched on"].map(
    lambda s: ", ".join(sorted(s)) if isinstance(s, set) else s)
mapping_for_spine = mapping_for_spine.rename(columns={"Gene stable ID": "BioMart Ensembl ID"})

# One row per gene: where several Hansen rows reached a gene, their values are joined with "; "
def combine_rows(values):
    if len(values) == 1:
        return values.iloc[0]
    return "; ".join(values.fillna("-").astype(str))

mapping_one_row_per_gene = (mapping_for_spine
    .sort_values("Hansen_idx")
    .groupby("BioMart Ensembl ID", as_index=False)
    .agg(combine_rows))

# MitoMammal's ID column is the spine; everything else attaches to it
mitomammal_trimmed = MitoMammal_BAT.rename(columns={"Gene stable ID": "MitoMammal Ensembl ID"})

mitomammal_trimmed = mitomammal_trimmed.merge(
    MitoMammal_Heart, left_on="MitoMammal Ensembl ID", right_on="Gene stable ID",
    how="left", validate="one_to_one").drop(columns="Gene stable ID")

mitomammal_trimmed = mitomammal_trimmed.merge(
    mapping_one_row_per_gene, left_on="MitoMammal Ensembl ID", right_on="BioMart Ensembl ID",
    how="left", validate="one_to_one")

# Label the MitoMammal genes that no Hansen row mapped to
mitomammal_trimmed["All identifiers"] = mitomammal_trimmed["All identifiers"].fillna(
    "No Hansen row mapped to this gene")

print("Rows:", len(mitomammal_trimmed))
print("ID column identical to MitoMammal:",
      mitomammal_trimmed["MitoMammal Ensembl ID"].equals(MitoMammal_BAT["Gene stable ID"]))
print("Genes with no Hansen mapping:", mitomammal_trimmed["BioMart Ensembl ID"].isna().sum())

Rows: 6652
ID column identical to MitoMammal: True
Genes with no Hansen mapping: 9


In [74]:
with pd.ExcelWriter(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\mitomammal_trimmed_mapping.xlsx") as writer:
    mitomammal_trimmed.to_excel(writer, sheet_name="MitoMammal genes", index=False)

In [75]:
with pd.ExcelWriter(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\merge_mapping.xlsx") as writer:
    merge_back_mapping.to_excel(writer, sheet_name="Full mapping", index=False)

In [76]:
print("Full mapping rows:", len(full_mapping_output))
print("Trimmed to MitoMammal rows:", len(mitomammal_mapping))
print("Total Ensembl Gene IDs in MitoMammal:", len(MitoMammal_BAT))
print("Distinct Ensembl genes:", mitomammal_mapping["MitoMammal Ensembl ID"].nunique())
print("MitoMammal genes with no Hansen mapping:",
      len(set(MitoMammal_BAT["Gene stable ID"]) - set(mitomammal_mapping["MitoMammal Ensembl ID"])))

Full mapping rows: 7989
Trimmed to MitoMammal rows: 7046
Total Ensembl Gene IDs in MitoMammal: 6652
Distinct Ensembl genes: 6643
MitoMammal genes with no Hansen mapping: 9


In [77]:
print(len(merge_back_mapping))
print(merge_back_mapping["Hansen_idx"].duplicated().sum())

7989
357


In [78]:
missing_genes = sorted(set(MitoMammal_BAT["Gene stable ID"]) - set(mitomammal_mapping["MitoMammal Ensembl ID"]))
print(len(missing_genes))
print(missing_genes)

9
['ENSMUSG00000082029', 'ENSMUSG00000087444', 'ENSMUSG00000091800', 'ENSMUSG00000092116', 'ENSMUSG00000095742', 'ENSMUSG00000098559', 'ENSMUSG00000106256', 'ENSMUSG00000114515', 'ENSMUSG00000121332']


In [79]:
# Check the exact header text of the column made in Excel
cols = pd.read_excel(
    r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\Files used to verify method of integration\proteinGroups.xlsx",
    nrows=0
).columns.tolist()
print([c for c in cols if "SUM" in c])

['SUM Razor + unique peptides SKM_']


In [80]:
# Duplicates: Ensembl genes reached by more than one Hansen row carrying different peptide counts.
# .transform("nunique") counts the distinct SKM values per gene but writes the answer back onto every row,
# so the result aligns with the table rather than being a separate summary.
values_per_gene = mitomammal_mapping.groupby("MitoMammal Ensembl ID")["Sum_Peptides SKM"].transform("nunique")

duplicates = mitomammal_mapping[values_per_gene > 1].sort_values(
    ["MitoMammal Ensembl ID", "Hansen_idx"])

print("Rows involved in a duplicate:", len(duplicates))
print("Genes affected:", duplicates["MitoMammal Ensembl ID"].nunique())

# Sheet 2: the duplicates where more than one competing row carries a nonzero value.
# These are the genes that genuinely need a decision rule. Where only one row is nonzero, the choice is
# obvious; where all rows are zero, the gene is not in this set at all, since it has only one distinct value.
nonzero_per_gene = duplicates.groupby("MitoMammal Ensembl ID")["Sum_Peptides SKM"].transform(
    lambda s: (s > 0).sum())

duplicates_nonzero = duplicates[nonzero_per_gene > 1]

print("Rows:", len(duplicates_nonzero))
print("Genes affected:", duplicates_nonzero["MitoMammal Ensembl ID"].nunique())



# Sheet 3: every duplicate row with the MaxQuant evidence columns attached.
# Sum_Peptides counts shared peptides in every group that contains them, so it cannot distinguish a real
# protein group from a subset of one. The razor and unique counts can, because MaxQuant already assigned
# each ambiguous peptide to a single group. The three flags mark groups that are not valid candidates.
# Built from duplicates rather than duplicates_nonzero, so all 207 rows are available for selection.
evidence_cols = ['Razor + unique peptides', 'Unique peptides',
                 'Reverse', 'Potential contaminant',
                 'Only identified by site', 'Q-value', 'Score',
                 'SUM Razor + unique peptides SKM_']

Hansen_evidence = pd.read_excel(
    r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\Files used to verify method of integration\proteinGroups.xlsx",
    usecols=evidence_cols
)

print("Evidence table rows:", len(Hansen_evidence))

duplicates_evidence = duplicates.merge(
    Hansen_evidence[evidence_cols],
    left_on="Hansen_idx",
    right_index=True,
    how="left"
)

print("Evidence sheet rows:", len(duplicates_evidence))
print("Rows with no evidence match:",
      duplicates_evidence["Razor + unique peptides"].isna().sum())

with pd.ExcelWriter(r"C:\Users\cayle\OneDrive\Documents\Master's Reserach\duplicate_genes.xlsx") as writer:
    duplicates.to_excel(writer, sheet_name="All duplicates", index=False)
    duplicates_nonzero.to_excel(writer, sheet_name="Several nonzero", index=False)
    duplicates_evidence.to_excel(writer, sheet_name="Evidence", index=False)

Rows involved in a duplicate: 207
Genes affected: 86
Rows: 158
Genes affected: 62
Evidence table rows: 7632
Evidence sheet rows: 207
Rows with no evidence match: 0


In [81]:
print(MitoMammal_BAT[MitoMammal_BAT["Gene stable ID"].isin(
    ["ENSMUSG00000030695", "ENSMUSG00000114515"])])

          Gene stable ID  Sum Peptide Counts MitoMammal BAT
2429  ENSMUSG00000114515                                104


In [82]:
# MitoMammal assigned Aldoa’s peptides to `ENSMUSG00000114515` (`Gm57854`), a predicted gene (automated software liekly) that overlaps the actual Aldoa gene.
#  The canonical Aldoa gene, `ENSMUSG00000030695`, was not included in their gene list.

# My mapping assigned Aldoa’s peptides to `ENSMUSG00000030695`, the canonical Aldoa gene associated with the reviewed Swiss-Prot ID `P05064`. T
# This is not recognised by the MitoMammal model.

# Therefore, for compatibility with MitoMammal, Aldoa’s SKM value of 131 should be assigned to their ID, `ENSMUSG00000114515`. The substitution should also be documented so that the difference between the two gene IDs is clearly recorded rather than silently changed.




#ENSMUSG00000114515 (Gm57854): a predicted gene model: annotation software found gene-like structure on chromosome 7 at 126,393,858–126,408,507 and assigned it an ID. 
# No curated name, no reviewed protein entry, only the unreviewed TrEMBL accession `A0A140LJ28`. 
# This is the ID MitoMammal used.

#ENSMUSG00000030695 (Aldoa): the curated gene: aldolase A, fructose-bisphosphate, a characterised glycolytic enzyme at 126,399,962–126,408,280 on the same chromosome, overlapping the model above. 
# Carries the reviewed Swiss-Prot accession `P05064` plus seven TrEMBL accessions. 
# This is the ID my pipeline used.

# So two Ensembl gene models sitting on the same stretch of DNA, describing the same protein. 
# One is a prediction with an unreviewed accession and the other is the named, curated gene. 
# MitoMammal's list contains only the first.


In [83]:
# Load the two files saved from Excel. Neither has a header row,
# so names are supplied.
folder = r"C:\Users\cayle\OneDrive\Documents\Master's Reserach"

final_SKM = pd.read_excel(
    folder + r"\SKM Data Before 86 Resolved Cases.xlsx",
    header=None, names=["Ensembl ID", "SKM"])

resolved_86 = pd.read_excel(
    folder + r"\SKM Resolved 86 Cases Only.xlsx",
    header=None, names=["Ensembl ID", "SKM"])


# The resolved file reads as 89 rows, not 86. Rows 87 and 88 are
# empty, and B89 holds =COUNT(B1:B86). pandas reads the result of
# that formula (86) as a value with no Ensembl ID beside it.
# Show, then remove, every row with a blank Ensembl ID.
blank_id_rows = resolved_86[resolved_86["Ensembl ID"].isna()]
print("Blank-ID rows removed:", len(blank_id_rows))
resolved_86 = resolved_86.dropna(subset=["Ensembl ID"])

# Remove exact copies (same ID and value), in case a row was pasted twice.
resolved_86 = resolved_86.drop_duplicates()

# An ID still listed twice was given two different values, which
# can only be corrected in the Excel file. raise stops the cell here.
repeated = resolved_86[resolved_86["Ensembl ID"].duplicated(keep=False)]
if len(repeated) > 0:
    print(repeated)
    raise ValueError("These IDs have two different values. "
                     "Fix them in the resolved Excel file.")


# The resolved file as a lookup: give it an Ensembl ID and it
# returns the value I selected.
resolved_lookup = resolved_86.set_index("Ensembl ID")["SKM"]

not_found = set(resolved_lookup.index) - set(final_SKM["Ensembl ID"])
print("Resolved genes:", len(resolved_lookup))
print("Resolved IDs not in the before file:", len(not_found))


# Every selected value should be one of the values the gene
# originally carried. The before cell is split at "; " into its
# separate values, and the selected value is checked against them.
before_values = final_SKM.set_index("Ensembl ID")["SKM"]

not_original = []
for gene, selected in resolved_lookup.items():
    if gene in before_values.index:
        options = str(before_values[gene]).split("; ")
        options = [float(v) for v in options]
        if float(selected) not in options:
            not_original.append(gene)

print("Selected values not among the originals:", len(not_original))


# Override: replace the SKM value only in the rows whose Ensembl ID
# is in the lookup. No row is sorted or moved.
is_manual = final_SKM["Ensembl ID"].isin(resolved_lookup.index)
final_SKM.loc[is_manual, "SKM"] = (
    final_SKM.loc[is_manual, "Ensembl ID"].map(resolved_lookup))
print("Rows overridden:", is_manual.sum())


# Collapse combined cells where every value is the same,
# e.g. "0; 0" becomes "0". set() keeps only the distinct values.
def collapse_identical(value):
    if isinstance(value, str) and "; " in value:
        parts = set(value.split("; "))
        if len(parts) == 1:
            return parts.pop()
    return value

is_combined = final_SKM["SKM"].astype(str).str.contains("; ")
print("Combined cells to collapse:", is_combined.sum())
final_SKM["SKM"] = final_SKM["SKM"].map(collapse_identical)


# Aldoa substitution: MitoMammal uses ENSMUSG00000114515 for Aldoa,
# so it receives Aldoa's SKM value of 131. Only filled if blank.
aldoa_row = final_SKM["Ensembl ID"] == "ENSMUSG00000114515"
aldoa_row = aldoa_row & final_SKM["SKM"].isna()
final_SKM.loc[aldoa_row, "SKM"] = 131
print("Aldoa value filled:", aldoa_row.sum())


# Final checks before saving.
still_combined = final_SKM[
    final_SKM["SKM"].astype(str).str.contains("; ")]
blank_rows = final_SKM[final_SKM["SKM"].isna()]
mm_ids = MitoMammal_BAT["Gene stable ID"]

print("Total rows:", len(final_SKM))
print("Same order as MitoMammal:", final_SKM["Ensembl ID"].equals(mm_ids))
print("Rows still combined:", len(still_combined))
print("Blank rows left:", len(blank_rows))
print(blank_rows["Ensembl ID"].tolist())


# Save only if every row holds a single value.
# pd.to_numeric turns the collapsed text values ("0") into numbers.
# header=False and index=False match MitoMammal's format.
if len(still_combined) == 0:
    final_SKM["SKM"] = pd.to_numeric(final_SKM["SKM"])
    with pd.ExcelWriter(folder + r"\final SKM dataset.xlsx") as writer:
        final_SKM.to_excel(writer, sheet_name="final SKM dataset",
                           header=False, index=False)
    print("Saved: final SKM dataset.xlsx")
else:
    print("NOT saved. These genes still need a decision:")
    print(still_combined)

# Blank-ID rows removed: 3
# Resolved genes: 86
# Resolved IDs not in the before file: 0
# Selected values not among the originals: 0
# Rows overridden: 86
# Combined cells to collapse: 243
# Aldoa value filled: 1
# Total rows: 6652
# Same order as MitoMammal: True
# Rows still combined: 0
# Blank rows left: 8
# Saved: final SKM dataset.xlsx

Blank-ID rows removed: 0
Resolved genes: 86
Resolved IDs not in the before file: 0
Selected values not among the originals: 0
Rows overridden: 86
Combined cells to collapse: 243
Aldoa value filled: 1
Total rows: 6652
Same order as MitoMammal: True
Rows still combined: 0
Blank rows left: 8
['ENSMUSG00000091800', 'ENSMUSG00000106256', 'ENSMUSG00000121332', 'ENSMUSG00000095742', 'ENSMUSG00000087444', 'ENSMUSG00000082029', 'ENSMUSG00000098559', 'ENSMUSG00000092116']
Saved: final SKM dataset.xlsx
